In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

df = pd.read_csv("data/raw_loan_dataset.csv")
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [10]:
# import os

# print(os.listdir("data"))

['.ipynb_checkpoints', 'raw_loan_dataset.csv.xlsx']


In [12]:

print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


In [15]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


In [20]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df.head()


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


In [21]:
# 4) Impute missing values
# --------------------------------
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


In [22]:
# 5) Remove duplicates
# --------------------------------
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")
df.head()


Dropped duplicates: 103 -> 100 rows


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


In [24]:
# --------------------------------
# 6) IQR capping on numeric columns
# L3: clip extreme values to the IQR fence — same idea as house data-processing.py
# --------------------------------
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
    
low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)



,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


In [25]:
df["Approved"] = df["Approved"].map({"Yes":1,"No":0})
df["HasCollateral"] = df["HasCollateral"].map({"Yes":1,"No":0})
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes":1,"No":0})
df.head()


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,1,0,0
1,96482.0,524.0,15.0,11200.0,1,0,1
2,28478.0,638.0,5.4,12100.0,0,0,1
3,25851.0,561.0,17.6,7000.0,1,0,1
4,38396.0,527.0,9.8,10700.0,0,0,1


In [27]:
# Feature Engineering
df["DebtToIncome"] = df["LoanAmount"] / df["Income"]
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,108810.0,537.0,1.1,25800.0,1,0,0,0.237111,51814.285714
1,96482.0,524.0,15.0,11200.0,1,0,1,0.116084,6030.125000
2,28478.0,638.0,5.4,12100.0,0,0,1,0.424889,4449.687500
3,25851.0,561.0,17.6,7000.0,1,0,1,0.270783,1389.838710
4,38396.0,527.0,9.8,10700.0,0,0,1,0.278675,3555.185185


In [28]:
# Scaling (MinMaxScaler)
scaler = MinMaxScaler()

cols = ["Income","CreditScore","LoanAmount","EmploymentYears"]
df[cols] = scaler.fit_transform(df[cols])
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.585174,0.121560,0.024490,0.339800,1,0,0,0.237111,51814.285714
1,0.498215,0.091743,0.591837,0.101287,1,0,1,0.116084,6030.125000
2,0.018530,0.353211,0.200000,0.115989,0,0,1,0.424889,4449.687500
3,0.000000,0.176606,0.697959,0.032673,1,0,1,0.270783,1389.838710
4,0.088490,0.098624,0.379592,0.093118,0,0,1,0.278675,3555.185185


In [29]:
df.to_csv("data/clean_loan_dataset.csv", index=False)
print("Saved clean dataset!")


Saved clean dataset!
